In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, learning_curve, cross_val_score, KFold, GridSearchCV
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report,f1_score, recall_score
from sklearn.neighbors import KNeighborsClassifier
from imblearn.over_sampling import SMOTE, RandomOverSampler

%matplotlib inline
import seaborn as sns
import matplotlib.pyplot as plt
plt.style.use('seaborn-whitegrid')
from warnings import filterwarnings
filterwarnings('ignore')

## Stroke Prediction Dataset

In [ ]:
# Stroke dataset
stroke_df = pd.read_csv('./healthcare-dataset-stroke-data.csv')

# Encode categorical features
stroke_df['gender'] = LabelEncoder().fit_transform(stroke_df['gender'])
stroke_df['ever_married'] = LabelEncoder().fit_transform(stroke_df['ever_married'])
stroke_df['Residence_type'] = LabelEncoder().fit_transform(stroke_df['Residence_type'])
stroke_df = pd.get_dummies(data=stroke_df, columns=['smoking_status', 'work_type'])

# fill in missing data
stroke_df['bmi'].fillna(stroke_df['bmi'].median(), inplace=True)

# scale numeric features
columns = ['avg_glucose_level', 'bmi', 'age']
scaler = StandardScaler()
stroke_df[columns] = pd.DataFrame(scaler.fit_transform(
    stroke_df[columns].values), columns=columns, index=stroke_df.index)

# Drop uneeded data
stroke_df.drop(['id'], axis=1, inplace=True)

X = stroke_df.drop(columns=['stroke'])
X = X[:].values
y = stroke_df['stroke'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=555, shuffle=True) 

ax, fig = plt.subplots(figsize=[15,7])
sns.countplot(x = y, data = stroke_df)

# Oversample with Smote
sampler = SMOTE(random_state=123)
X_train, y_train = sampler.fit_resample(X_train, y_train)

stroke_data = X_train, X_test, y_train, y_test


## EMG Gesture Dataset

In [ ]:
# Muscle Activity Gestures Dataset

gesture_df1 = pd.read_csv("./0.csv", header=None)
gesture_df2 = pd.read_csv("./1.csv", header=None)
gesture_df3 = pd.read_csv("./2.csv", header=None)
gesture_df4 = pd.read_csv("./3.csv", header=None)

# Prep data
gesture_df = pd.concat([gesture_df1, gesture_df2, gesture_df3, gesture_df4], axis=0)

ax, fig = plt.subplots(figsize=[15,7])
sns.countplot(x = 64, data = gesture_df)


X = gesture_df.drop(columns=[64])
X = X[:].values
y = gesture_df[64].values


print(len(y))

# Scale X features
# X = StandardScaler().fit_transform(X)

# sns.countplot(x = gesture_df[:,64], data = gesture_df)

gesture_data = train_test_split(X, y, test_size=0.30, random_state=555, shuffle=True)

In [ ]:


def get_all_results(y_test, y_pred, type):
	if type == 'gesture':
		f1 = f1_score(y_test, y_pred,average='micro')
		recall = recall_score(y_test, y_pred,average='micro')
	else:
		f1 = f1_score(y_test, y_pred)
		recall = recall_score(y_test, y_pred)
	accuracy = accuracy_score(y_test, y_pred)

	return {
		'f1': f1,
		'accuracy_score': accuracy,
		'recall': recall
	}


In [ ]:
# Generate confusion matrix
def show_confusion_matrix(y_test, y_pred, type):

    if type == 'stroke':
        yticklabels = ['No stroke', 'Stroke']
        xticklabels = ['Predicted no stroke', 'Predicted stroke']
    elif type == 'gesture':
        yticklabels = ['Rock', 'Scissors', 'Paper', 'OK']
        xticklabels = ['Predicted rock', 'Predicted scissors', 'Predicted Paper', 'Predicted OK']

    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, cmap='Blues', annot=True, fmt='d', linewidths=5, cbar=False, annot_kws={'fontsize': 15},
                yticklabels=yticklabels, xticklabels=xticklabels)
    plt.yticks(rotation=0)
    plt.show()


In [ ]:
def gen_learning_curve(algo, title, X, y, cv, train_sizes=np.linspace(.1, 1.0, 5), scoring=None):

    train_sizes, train_scores, test_scores = learning_curve(
        algo, X, y, cv=cv, train_sizes=train_sizes, scoring=scoring, n_jobs=-1)
    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    test_scores_std = np.std(test_scores, axis=1)

    plt.figure(figsize=(10, 10))
    plt.tick_params(labelsize=14)
    plt.title(title)
    plt.xlabel("Training examples")
    plt.ylabel("Score")
    plt.grid()

    plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.1, color="r")
    plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                     test_scores_mean + test_scores_std, alpha=0.1, color="g")
    plt.plot(train_sizes, train_scores_mean, 'o-',
             color="r", label="Training score")
    plt.plot(train_sizes, test_scores_mean, 'o-',
             color="g", label="Cross-validation score")

    plt.legend(loc="best")
    plt.show()


# Decision Tree

In [ ]:
# Decision tree
# Stroke: pruning helped!(ccp_alpha) wth criterion=’entropy’


def decision_tree(type, scoring, cv, data, params = None):
    params = params or {
        'ccp_alpha': [0.003, 0.00001, 0.3, 0.1, 0.2, 0.5],
        "criterion": ["gini", "entropy"],
        "max_depth": [None, 3, 5, 10, 20],
        "min_samples_split": [2, 4, 10],
    }

    X_train, X_test, y_train, y_test = data
    model = GridSearchCV(
        DecisionTreeClassifier(),
        params,
        n_jobs=-1,
        return_train_score=True,
        scoring=scoring,
        cv=cv,
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(model.best_score_, model.best_params_,)
    print(classification_report(y_test, y_pred))
    show_confusion_matrix(y_test, y_pred, type)
    gen_learning_curve(
        DecisionTreeClassifier(**(model.best_params_)),
        f'Learning Curve (Descision Tree {scoring})',
        X_train,
        y_train,
        cv=cv,
        scoring=scoring
    )

    results = get_all_results(y_test, y_pred, type)
    return results, model.best_params_

dt_results_stroke_f1, best_stroke_dt_f1 = decision_tree('stroke', 'f1', 10, stroke_data)
dt_results_stroke_recall, best_stroke_dt_recall = decision_tree('stroke', 'recall', 10, stroke_data)
dt_results_gesture, best_gesture_dt = decision_tree('gesture', None, 10, gesture_data)


# Neural network

In [ ]:
# Neural network

def neural_net(type, scoring, cv, data, params=None):
    X_train, X_test, y_train, y_test = data
    params = params or {
        'max_iter': [3000],
        'activation': ['identity', 'logistic', 'tanh', 'relu'],
        'solver': ['lbfgs', 'sgd', 'adam'],
        'hidden_layer_sizes': [
            (1,), (2,), (5,), (11,), (21,)
        ]
    }

    # scale all features for NN
    scaler = StandardScaler()
    scaler.fit(X_train)
    X_train = scaler.transform(X_train)
    X_test = scaler.transform(X_test)

    model = GridSearchCV(
        MLPClassifier(),
        params,
        scoring=scoring,
        cv=cv,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Results
    print(model.best_score_, model.best_params_,)
    print(classification_report(y_test, y_pred))
    show_confusion_matrix(y_test, y_pred, type)
    gen_learning_curve(
        MLPClassifier(**(model.best_params_)),
        f'Learning Curve (NN {scoring})',
        X_train,
        y_train,
        cv=cv,
        scoring=scoring
    )

    results = get_all_results(y_test, y_pred, type)
    return results


nn_results_stroke_f1 = neural_net('stroke', 'f1', 10, stroke_data)
nn_results_stroke_recall = neural_net('stroke', 'recall', 10, stroke_data)
nn_result_gesture = neural_net('gesture', None, 10, gesture_data)


# Boosted Decision Trees

In [ ]:
# Boosted Decision Trees

def boosted_trees(type, scoring, cv, data, params=None):
    params = params or {
        "n_estimators": [5, 10, 15, 20, 25, 50, 75, 100],
        "learning_rate": [0.001, 0.01, 0.1, 0.2, 1.],
    }

    X_train, X_test, y_train, y_test = data


    model = GridSearchCV(
        GradientBoostingClassifier(),
        params,
        scoring=scoring,
        cv=cv,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(model.best_score_, model.best_params_,)
    print(classification_report(y_test, y_pred))
    show_confusion_matrix(y_test, y_pred, type)
    gen_learning_curve(
        GradientBoostingClassifier(**(model.best_params_)),
        f'Learning Curve (Boosted {scoring})',
        X_train,
        y_train,
        cv=cv,
        scoring=scoring
    )

    results = get_all_results(y_test, y_pred, type)
    return results



boost_stroke_result_f1 = boosted_trees('stroke', 'f1', 10, stroke_data)
boost_stroke_result_recall = boosted_trees('stroke', 'recall', 10, stroke_data)
boost_gesture_result = boosted_trees('gesture', None, 10, gesture_data)

# Support Vector Machines

In [ ]:
# Support Vector Machines

def svm(type, scoring, cv, data, params = None):
    params = params or {
        'kernel': ['rbf', 'sigmoid', 'linear']
    }

    X_train, X_test, y_train, y_test = data
    model = GridSearchCV(
        SVC(),
        params,
        scoring=scoring,
        n_jobs=-1,
        cv=cv,
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f'---- dataset: {type}, scoring: {scoring} ---')
    print(model.best_score_, model.best_params_,)
    print(classification_report(y_test, y_pred))
    show_confusion_matrix(y_test, y_pred, type)
    gen_learning_curve(
        SVC(**(model.best_params_)),
        f'Learning Curve (SVM {scoring})',
        X_train,
        y_train,
        cv=cv,
        scoring=scoring
    )
    print(f'-------')

    results = get_all_results(y_test, y_pred, type)
    return results


svm_result1 = svm('stroke', 'f1', 10, stroke_data)
svm_result2 = svm('stroke', 'recall', 10, stroke_data)
svm_result3 = svm('gesture', None, 10, gesture_data)

# KNN

In [ ]:
# KNN

def knn(type, scoring, cv, data, params = None):
    params = params or {
        'weights': ('uniform', 'distance'),
        'n_neighbors': [3, 7, 10, 30],
    }

    X_train, X_test, y_train, y_test = data

    model = GridSearchCV(
        KNeighborsClassifier(),
        params,
        n_jobs=-1,
        scoring=scoring,
        return_train_score=True,
        cv=cv,
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f'---- dataset: {type}, scoring: {scoring} ---')
    print(model.best_score_, model.best_params_,)
    print(classification_report(y_test, y_pred))
    show_confusion_matrix(y_test, y_pred, type)
    scoring_name = scoring or ''
    gen_learning_curve(
        KNeighborsClassifier(**(model.best_params_)),
        f'Learning Curve (KNN {scoring_name})',
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
    )
    print(f'-------')

    results = get_all_results(y_test, y_pred, type)
    return results


knn1 = knn('stroke', 'f1', 20, stroke_data)
knn2 = knn('stroke', 'recall', 30, stroke_data)
knn3 = knn('gesture', None, 10, gesture_data)